# CASE 4: Global Pool Evaluation, Ensemble Architecture & ADASYN Harmonization


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import ADASYN
from collections import Counter

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, f1_score, matthews_corrcoef



## 1. Data Harmonization
Load CICIDS2017 and CIC-IDS-2018, intersect identical features, and merge into a Global Unbalanced Pool.


In [3]:
# Mapping Dictionary: {2018_Name : 2017_Name}
col_mapping = {
    'Dst Port': 'Destination Port',
    'Tot Fwd Pkts': 'Total Fwd Packets',
    'Tot Bwd Pkts': 'Total Backward Packets',
    'TotLen Fwd Pkts': 'Total Length of Fwd Packets',
    'TotLen Bwd Pkts': 'Total Length of Bwd Packets',
    'Fwd Pkt Len Max': 'Fwd Packet Length Max',
    'Fwd Pkt Len Min': 'Fwd Packet Length Min',
    'Fwd Pkt Len Mean': 'Fwd Packet Length Mean',
    'Fwd Pkt Len Std': 'Fwd Packet Length Std',
    'Bwd Pkt Len Max': 'Bwd Packet Length Max',
    'Bwd Pkt Len Min': 'Bwd Packet Length Min',
    'Bwd Pkt Len Mean': 'Bwd Packet Length Mean',
    'Bwd Pkt Len Std': 'Bwd Packet Length Std',
    'Flow Byts/s': 'Flow Bytes/s',
    'Flow Pkts/s': 'Flow Packets/s',
    'Fwd IAT Tot': 'Fwd IAT Total',
    'Bwd IAT Tot': 'Bwd IAT Total',
    'Fwd Header Len': 'Fwd Header Length',
    'Bwd Header Len': 'Bwd Header Length',
    'Pkt Len Min': 'Min Packet Length',
    'Pkt Len Max': 'Max Packet Length',
    'Pkt Len Mean': 'Packet Length Mean',
    'Pkt Len Std': 'Packet Length Std',
    'SYN Flag Cnt': 'SYN Flag Count',
    'PSH Flag Cnt': 'PSH Flag Count',
    'ACK Flag Cnt': 'ACK Flag Count',
    'Pkt Size Avg': 'Average Packet Size',
    'Fwd Seg Size Avg': 'Avg Fwd Segment Size',
    'Bwd Seg Size Avg': 'Avg Bwd Segment Size',
    'Subflow Fwd Byts': 'Subflow Fwd Bytes',
    'Subflow Bwd Byts': 'Subflow Bwd Bytes',
    'Init Fwd Win Byts': 'Init_Win_bytes_forward',
    'Init Bwd Win Byts': 'Init_Win_bytes_backward',
    'Fwd Act Data Pkts': 'act_data_pkt_fwd',
    'Fwd Seg Size Min': 'min_seg_size_forward'
}



# 1. Load Headers
header_2017 = pd.read_csv('CICIDS2017_preprocessed.csv', nrows=0).columns.tolist()
header_2018_raw = pd.read_csv('CIC_IDS_2018_Preprocessed_Combined_2.csv', nrows=0).columns.tolist()

# 2. Rename 2018 Headers locally to match 2017
# We create a list of what the 2018 columns WOULD be called if they were 2017 names
header_2018_mapped = [col_mapping.get(col, col) for col in header_2018_raw]

# 3. Find the Intersection
# These are the 2017-style names that exist in both
common_cols_2017_style = sorted(list(set(header_2017) & set(header_2018_mapped)))

# We need the REVERSE mapping to tell the 2018 loader which RAW columns to grab
reverse_mapping = {v: k for k, v in col_mapping.items()}
cols_to_load_from_2018 = [reverse_mapping.get(col, col) for col in common_cols_2017_style]

print(f"Success! Harmonized columns found: {len(common_cols_2017_style)}")

# 4. Process 2018 in Chunks
chunks_2018 = pd.read_csv(
    'CIC_IDS_2018_Preprocessed_Combined_2.csv', 
    usecols=cols_to_load_from_2018, # Load 2018 using its own abbreviations
    iterator=True, 
    chunksize=100000, 
    on_bad_lines='skip'
)

processed_chunks = []
for chunk in chunks_2018:
    # RENAME the chunk columns to 2017 style immediately after loading
    chunk.rename(columns=col_mapping, inplace=True)
    
    # Standardize Label & Downcast
    chunk['Label'] = pd.to_numeric(chunk['Label'], errors='coerce').fillna(-1).astype(np.int8)
    chunk = chunk[chunk['Label'] != -1]
    
    float_cols = chunk.select_dtypes(include=['float64']).columns
    chunk[float_cols] = chunk[float_cols].astype(np.float32)
    processed_chunks.append(chunk)

df_2018 = pd.concat(processed_chunks, ignore_index=True)

# 5. Load 2017 and Merge
df_2017 = pd.read_csv('CICIDS2017_preprocessed.csv', usecols=common_cols_2017_style)
df_2017['Label'] = pd.to_numeric(df_2017['Label'], errors='coerce').fillna(-1).astype(np.int8)

df_global = pd.concat([df_2017, df_2018], ignore_index=True)
print(f"Final Global Pool Shape: {df_global.shape}")
df_global.to_csv("Global_Pool_Unbalanced_Train.csv", index=False)

Success! Harmonized columns found: 56
Final Global Pool Shape: (3533515, 56)


## 2. Phase 1: Unbalanced Evaluation (70/30 Split)
Perform a 70/30 stratified split, train 6 models including an Ensemble architecture natively.


In [1]:
X = df_global.drop(columns=['Label'])
y = df_global['Label']

# Depending on total global pool massive dimensions, memory scaling is implemented here safely if needed natively.
# For full processing we maintain 100% of data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

print(f"Training Unbalanced split : {X_train.shape}")
print(f"Testing  Unbalanced split : {X_test.shape}")

models = {
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, random_state=42)
}

# Define Stacked Ensemble explicitly
estimators = [
    ('nb', GaussianNB()),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42))
]

models['Stacked Ensemble'] = StackingClassifier(
    estimators=estimators, 
    final_estimator=LogisticRegression(max_iter=1000),
    cv=3,
    n_jobs=-1
)

results_unbalanced = {}

print("\n--- Phase 1: Training Unbalanced Models ---")
for name, model in models.items():
    print(f"Training base framework -> {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results_unbalanced[name] = {'Macro F1': macro_f1, 'MCC': mcc}
    print(f" [Results] {name} -> Macro F1: {macro_f1:.4f} | MCC: {mcc:.4f}")
    print(f" Detailed Report for {name}:\n", classification_report(y_test, y_pred))



NameError: name 'df_global' is not defined

## 3. Phase 2: ADASYN Balancing
Apply ADASYN synthetics exclusively to the training split, keeping testing pure.


In [ ]:
print("Class distribution Before ADASYN:")
print(y_train.value_counts())

# Safety measure: ADASYN natively requires minimum > 5 samples per class. Filter outliers.
c = Counter(y_train)
rare_classes = [k for k,v in c.items() if v < 6]
if rare_classes:
    print(f"Filtering out ultra-rare classes {rare_classes} natively too small for ADASYN generation")
    mask = ~y_train.isin(rare_classes)
    X_train_filtered = X_train[mask]
    y_train_filtered = y_train[mask]
else:
    X_train_filtered = X_train
    y_train_filtered = y_train

print("\nApplying ADASYN (Adaptive Synthetic Over-sampling)...")
adasyn = ADASYN(random_state=42, n_neighbors=5)
X_train_bal, y_train_bal = adasyn.fit_resample(X_train_filtered, y_train_filtered)

print("\nClass distribution AFTER ADASYN:")
print(y_train_bal.value_counts())

df_bal = pd.DataFrame(X_train_bal, columns=X.columns)
df_bal['Label'] = y_train_bal

output_bal = "Global_Pool_Balanced_Train.csv"
print(f"\nExporting natively to {output_bal}...")
df_bal.to_csv(output_bal, index=False)
print("Balanced matrix export completed!")



## 4. Phase 3: Balanced Evaluation
Retrain strictly all 6 model forms globally on the harmonized ADASYN distribution.


In [ ]:
results_balanced = {}

# Clean test set from previously isolated rare anomalies natively to preserve matrices
if rare_classes:
    mask_test = ~y_test.isin(rare_classes)
    X_test_filtered = X_test[mask_test]
    y_test_filtered = y_test[mask_test]
else:
    X_test_filtered = X_test
    y_test_filtered = y_test

print("--- Phase 3: Training Balanced Models ---")
for name, model in models.items():
    print(f"Retraining {name} (Balanced)...")
    
    # Trigger fresh fit
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test_filtered)
    
    macro_f1 = f1_score(y_test_filtered, y_pred, average='macro')
    mcc = matthews_corrcoef(y_test_filtered, y_pred)
    
    results_balanced[name] = {'Macro F1': macro_f1, 'MCC': mcc}
    print(f" [Results] {name} -> Macro F1: {macro_f1:.4f} | MCC: {mcc:.4f}")



## 5. Summary Diagnostics & Feature Importances
Display direct comparative mapping output logic.


In [ ]:
# Compile Table
comparison_rows = []
for name in models.keys():
    comparison_rows.append({
        'Model': name,
        'Macro F1 (Unbalanced)': results_unbalanced[name]['Macro F1'],
        'MCC (Unbalanced)': results_unbalanced[name]['MCC'],
        'Macro F1 (Balanced)': results_balanced[name]['Macro F1'],
        'MCC (Balanced)': results_balanced[name]['MCC']
    })

df_comp = pd.DataFrame(comparison_rows)
print("\n================ MODEL COMPARISON TABLE ================\n")
print(df_comp.to_string(index=False))

# Identify Winners programmatically
best_unbalanced = max(results_unbalanced, key=lambda x: results_unbalanced[x]['MCC'])
best_balanced = max(results_balanced, key=lambda x: results_balanced[x]['MCC'])

print("\n✅ CONCLUSION LOGIC:")
print(f"🥇 Highest MCC (Unbalanced Phase) : -> {best_unbalanced} ({results_unbalanced[best_unbalanced]['MCC']:.4f})")
print(f"🥇 Highest MCC (Balanced Phase)   : -> {best_balanced} ({results_balanced[best_balanced]['MCC']:.4f})")

# Extract final Feature Importances
print("\n--- Stacked Ensemble Top Feature Importances ---")
final_model = models['Stacked Ensemble']
rf_base = final_model.named_estimators_['rf']
importances = rf_base.feature_importances_
feat_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Influential Features (Detected through internal RF learner logic):")
print(feat_df.head(10).to_string(index=False))

print("\nMeta-Classifier Base-Learner Component Contribution Mapping:")
lr_meta = final_model.final_estimator_
if lr_meta.coef_.ndim > 1:
    meta_weights = np.mean(np.abs(lr_meta.coef_), axis=0) # Take average influence over all dimensions
else:
    meta_weights = np.abs(lr_meta.coef_[0])
    
base_names = [name for name, _ in estimators]
for i, name in enumerate(base_names[:len(meta_weights)]):
    print(f" - {name} Internal Weight Influence: {meta_weights[i]:.4f}")



## Visualization of Results
The following cells generate graphical representations of the model comparisons, top feature importances, and meta-learner internal weight influences based on the earlier evaluations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# 1. Model Comparison Chart
data = {
    'Model': ['Naive Bayes', 'KNN', 'Decision Tree', 'Random Forest', 'XGBoost', 'Stacked Ensemble'],
    'Macro F1 (Unbalanced)': [0.090479, 0.906522, 0.924598, 0.939648, 0.927382, 0.937463],
    'MCC (Unbalanced)': [0.060545, 0.936173, 0.943601, 0.944283, 0.944558, 0.945242],
    'Macro F1 (Balanced)': [0.089902, 0.885330, 0.923476, 0.922185, 0.908210, 0.920611],
    'MCC (Balanced)': [0.059745, 0.932726, 0.943626, 0.943848, 0.943531, 0.944662]
}
df = pd.DataFrame(data)
df_melt = df.melt(id_vars=['Model'], var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=df_melt, x='Model', y='Score', hue='Metric', palette='viridis')
plt.title("CASE 4: Model Evaluation (Balanced vs Unbalanced Pools)", fontsize=16, fontweight='bold', pad=15)
plt.ylabel("Performance Score", fontsize=12)
plt.xlabel("Algorithm", fontsize=12)
plt.xticks(rotation=0, fontsize=11)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, shadow=True)
plt.ylim(0, 1.05)

for p in ax.patches:
    if p.get_height() > 0.1:
        ax.annotate(format(p.get_height(), '.3f'), 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'center', 
                    xytext = (0, 9), 
                    textcoords = 'offset points',
                    fontsize=8, rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
# 2. Top Features Chart
features = ['act_data_pkt_fwd', 'Subflow Bwd Bytes', 'Destination Port', 'Flow Packets/s', 'Bwd Header Length', 'Flow Bytes/s']
importance = [0.062693, 0.046837, 0.039990, 0.035798, 0.029482, 0.027294]

plt.figure(figsize=(10, 6))
sns.barplot(x=importance, y=features, palette='magma', hue=features, legend=False)
plt.title("Stacked Ensemble: Top Feature Influences", fontsize=16, fontweight='bold')
plt.xlabel("Gini Importance Score", fontsize=12)
plt.ylabel("Network Feature", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Meta Learner Weights Chart
weights = {'KNN Base': 0.1998, 'Decision Tree Base': 0.2172, 'Random Forest Base': 0.1306, 'XGBoost Base': 0.2560}
plt.figure(figsize=(8, 8))
colors = sns.color_palette('husl', len(weights))
explode = (0, 0, 0, 0.05)  # slightly explode the biggest influence (XGBoost)

plt.pie(weights.values(), labels=weights.keys(), autopct='%1.1f%%', startangle=140, colors=colors, explode=explode, shadow=True, textprops={'fontsize': 12, 'weight': 'bold'})
plt.title("Meta-Learner Internal Weight Allocation", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
